In [ ]:
from pathlib import Path
from mutools.plotting import run
import mutools.plotting as mp
from mutools.plotting.profit import ProfitPlotData, TraceType

mp.saver.configure(fmt="pdf", dpi=300, rasterized=True, ytick_precision=2, mark_hadj=0.055)
mp.use_style("rootlike")

INPUT = Path('/Users/mueller/Projects/spineosc')

# Set to a directory to write PDFs; leave None to display inline while working.
# Figures are staged here first; copy into /Users/mueller/Projects/spineosc_technote/figures/detector_data_access once checked.
SAVE_TO = INPUT / 'docs' / 'detector_access_v1.1.0'
SAVE_TO.mkdir(parents=True, exist_ok=True)

In [ ]:
# Detector-access data/MC comparisons, from xml/detector_data_access/.
#
# One PROfit output per detector x variable x channel mode, each carrying the
# full systematics suite. POT in those configs is deliberately bogus and the
# comparisons are area normalized, so NOTHING here shows an absolute rate: no
# POT watermark, and the legend quotes each subchannel as a percentage of the
# prediction rather than a yield.

DETECTORS = {
    # token: legend label
    'sbnd_run1':   'SBND Run 1',
    'icarus_run2': 'ICARUS Run 2',
}

CHANNELS = {
    # channel mode: (per-channel legend labels, filename tokens)
    'inclusive': ([None], ['incl']),
    'exclusive': (['Single-Proton', 'Multi-Proton'], ['singlep', 'multip']),
    # Coarse azimuth (x-y plane) thirds of the leading muon direction, split at
    # +/-pi/3. Boundaries come from the MC tertiles rounded to 60 degrees, and
    # are identical in both detectors so the panels compare directly.
    'azimuth': ([r'$\phi < -60^{\circ}$', r'$|\phi| < 60^{\circ}$', r'$\phi > +60^{\circ}$'],
                ['phineg', 'phimid', 'phipos']),
    # Eight equal-width slices of [-pi, pi], inclusive channel only. SBND keeps
    # the nominal binning; the ICARUS configs halve it (variable token carries
    # the _coarse suffix) because a few slices would otherwise have empty CV bins.
    'azimuth8': ([rf'slice {i}' for i in range(8)], [f'phi{i}' for i in range(8)]),
}

VARIABLES = {
    # token: (x-axis label, xlim)
    'muon_score':           ('Muon PID Score',      [0.30, 1.00]),
    'proton_score':         ('Proton PID Score',    [0.40, 1.00]),
    'muon_primary_score':   ('Muon Primary Score',  [0.80, 1.00]),
    'proton_primary_score': ('Proton Primary Score', [0.10, 1.00]),
    # Doubled binning, inclusive and exclusive only.
    'muon_score_fine':           ('Muon PID Score',       [0.30, 1.00]),
    'proton_score_fine':         ('Proton PID Score',     [0.40, 1.00]),
    'muon_primary_score_fine':   ('Muon Primary Score',   [0.80, 1.00]),
    'proton_primary_score_fine': ('Proton Primary Score', [0.10, 1.00]),
    # ICARUS azimuth8 uses halved binning under its own token.
    'muon_score_coarse':           ('Muon PID Score',       [0.30, 1.00]),
    'proton_score_coarse':         ('Proton PID Score',     [0.40, 1.00]),
    'muon_primary_score_coarse':   ('Muon Primary Score',   [0.80, 1.00]),
    'proton_primary_score_coarse': ('Proton Primary Score', [0.10, 1.00]),
    # Zoom windows on the detector features. The x windows differ by detector
    # (SBND cathode at x = 0; ICARUS at folded |x| = 210.12), so their labels
    # and ranges come from VARIABLE_OVERRIDES below.
    'vertexx_zoom':   ('Vertex X [cm]',     [-50.0, 50.0]),
    'muon_endx_zoom': ('Muon End X [cm]',   [-50.0, 50.0]),
    'vertexz_zoom':   ('Vertex Z [cm]',     [-50.0, 50.0]),
    'muon_endz_zoom': ('Muon End Z [cm]',   [-50.0, 50.0]),
    # SBND APA gap region (assumed at z = 250 cm), 4 cm bins.
    'vertexz_apagap':   ('Vertex Z [cm]',   [210.0, 290.0]),
    'muon_endz_apagap': ('Muon End Z [cm]', [210.0, 290.0]),
}

# (variable, detector) -> (xlabel, xlim), for variables whose window depends on
# the detector.
VARIABLE_OVERRIDES = {
    ('vertexx_zoom', 'icarus_run2'):   (r'|Vertex X| [cm]',   [160.12, 260.12]),
    ('muon_endx_zoom', 'icarus_run2'): (r'|Muon End X| [cm]', [160.12, 260.12]),
}


def axis_for(variable, detector):
    return VARIABLE_OVERRIDES.get((variable, detector), VARIABLES[variable])

# ---------------------------------------------------------------------------
# Y-AXIS RANGES -- set by hand, one entry per variable so the same variable is
# directly comparable across detectors and channel modes. Run the report cell
# below to see the largest value actually plotted. None falls back to
# matplotlib's autoscale, which makes panels disagree with each other.
# ---------------------------------------------------------------------------
YLIM = {
    'muon_score':           [0.0, 14.0],
    'proton_score':         [0.0, 45.0],
    'muon_primary_score':   [0.0, 45.0],
    'proton_primary_score': [0.0, 11.0],
    'muon_score_fine':           [0.0, 14.0],
    'proton_score_fine':         [0.0, 60.0],
    'muon_primary_score_fine':   [0.0, 45.0],
    'proton_primary_score_fine': [0.0, 16.0],
    'muon_score_coarse':           [0.0, 14.0],
    'proton_score_coarse':         [0.0, 45.0],
    'muon_primary_score_coarse':   [0.0, 45.0],
    'proton_primary_score_coarse': [0.0, 11.0],
    # Zoom panels are area-normalized densities over a narrow window, so their
    # scale depends on the detector; the per-detector values live in
    # YLIM_OVERRIDES below and these are only the fallback.
    'vertexx_zoom': [0.0, 0.027], 'muon_endx_zoom': [0.0, 0.027],
    'vertexz_zoom': [0.0, 0.023], 'muon_endz_zoom': [0.0, 0.023],
}

# Optional exceptions, keyed (variable,), (variable, detector) or
# (variable, detector, channel_index). Most specific wins.
#
# The zoom panels peak around 0.012-0.018, and autoscaling ran the tallest bins
# into the two-column legend. These give ~55% headroom above the tallest of
# (CV, band top, data + error), and are shared between the vertex and muon-end
# panels of a detector so the two are read on the same scale.
YLIM_OVERRIDES = {
    ('vertexx_zoom',   'sbnd_run1'):   [0.0, 0.027],   # peak 0.0134; 0.021 clipped the legend
    ('muon_endx_zoom', 'sbnd_run1'):   [0.0, 0.027],   # peak 0.0124; 0.021 clipped the legend
    ('vertexx_zoom',   'icarus_run2'): [0.0, 0.027],   # peak 0.0174
    ('muon_endx_zoom', 'icarus_run2'): [0.0, 0.027],   # peak 0.0155
    ('vertexz_zoom',   'icarus_run2'): [0.0, 0.023],   # peak 0.0137
    ('vertexz_apagap',   'sbnd_run1'): [0.0, 0.030],    # peak 0.0148; was 0.0065 (clipped); 0.030 clears the 5-row legend
    ('muon_endz_apagap', 'sbnd_run1'): [0.0, 0.030],    # peak 0.0148; was 0.0065 (clipped); 0.030 clears the 5-row legend
    ('muon_endz_zoom', 'icarus_run2'): [0.0, 0.023],   # peak 0.0147
}

# Legend corner and column count per variable.
LEGEND = {
    'muon_score':           ('upper left', 2),
    'proton_score':         ('upper left', 2),
    'muon_primary_score':   ('upper left', 2),
    'proton_primary_score': ('upper right', 2),
}
DEFAULT_LEGEND = ('upper left', 2)

# Subchannel order is the XML's declaration order reversed: mutools pairs the
# FIRST label with the LAST subchannel (see construct_proxy_stack).
SUBCHANNELS = [r'Target $\nu_\mu$ CC', r'Other $\nu_\mu$ CC', r'$\nu_e$ CC', r'$\nu$ NC',
               'Out-of-time Cosmics', 'In-time Cosmics']
SUBCHANNEL_COLORS = ['C0', 'C1', 'C2', 'C4', 'C3', 'C5']

CODE_VERSION = 'v3.0.4'
SELECTION_VERSION = 'v1.1.0'


def output_file(detector, variable, channels):
    """PROfit outputs. SBND: runs/da_v2 (v1.1.0 selection with the cathode-crosser and 4 cm muon end-point requirements, xml/detector_access_v2/).
    ICARUS Run 2: runs/dent_study (v1.1.0 cvrw + cvwgt, no SBND cuts)."""
    study = f'{variable}_{channels}'
    if detector == 'sbnd_run1':
        return INPUT / 'runs' / 'da_v2' / f'da2_sbnd_run1_{study}_v1_PROplot_histograms.root'
    return INPUT / 'runs' / 'dent_study' / f'dent_{detector}_{study}_v1_PROplot_histograms.root'


def ylim_for(variable, detector, channel):
    for key in ((variable, detector, channel), (variable, detector), (variable,)):
        if key in YLIM_OVERRIDES:
            return YLIM_OVERRIDES[key]
    return YLIM.get(variable)


In [ ]:
# What is available, and how tall each panel actually is -- use it to set YLIM
# above. Runs still in progress simply show as missing.
import numpy as np

print(f"{'file':52s} {'peak':>8s}  {'ylim in use':>16s}")
for variable in VARIABLES:
    for channels, (labels, _) in CHANNELS.items():
        for detector in DETECTORS:
            path = output_file(detector, variable, channels)
            stem = f'{detector}/{variable}_{channels}'
            if not path.exists():
                print(f'{stem:52s} {"MISSING":>8s}')
                continue
            d = ProfitPlotData(path, True)
            peaks = []
            for channel in range(len(labels)):
                cv = d.get_trace(f'0:0:0:{channel}:total:CV', TraceType.HIST_CONTENTS)
                band = d.get_trace(f'0:0:0:{channel}:total:CV', TraceType.HIST_ERROR_BAND)
                dat = d.get_trace(f'0:0:0:{channel}:total:DATA', TraceType.HIST_CONTENTS)
                derr = dat[:, 4] if dat.shape[1] > 4 else np.sqrt(np.maximum(dat[:, 3], 0.0))
                peaks.append(float(np.nanmax(np.concatenate([cv[:, 3], band[:, 1] + band[:, 3], dat[:, 3] + derr]))))
            print(f'{stem:52s} {max(peaks):8.2f}  {str(ylim_for(variable, detector, 0)):>16s}')


In [ ]:
# Render every available comparison. Missing outputs are skipped, so this can
# be re-run while the PROfit jobs are still going.
import shutil
import matplotlib.pyplot as plt

made = 0
for variable in VARIABLES:
    for channels, (channel_labels, channel_tokens) in CHANNELS.items():
        for detector, detector_label in DETECTORS.items():
            xlabel, xlim = axis_for(variable, detector)
            path = output_file(detector, variable, channels)
            if not path.exists():
                print(f'skipping {detector}/{variable}_{channels}: not produced yet')
                continue
            legend_loc, legend_ncol = LEGEND.get(variable, DEFAULT_LEGEND)
            for channel, channel_label in enumerate(channel_labels):
                ylim = ylim_for(variable, detector, channel)
                run({
                    'general': {
                        'input': str(path),
                        'output': str(Path(SAVE_TO)) if SAVE_TO else '',
                        'savefig': SAVE_TO is not None,
                        'code_version': CODE_VERSION,
                        'selection_version': SELECTION_VERSION,
                        'subchannels': SUBCHANNELS,
                        'detectors': [detector_label],
                        'channels': channel_labels,
                        # The inis use scale-by-width, so stored contents are
                        # densities; this recovers the counts the legend
                        # percentages are computed from.
                        'scale_by_width': 'enabled',
                        'counter_index': 4,   # Count is var4 in these configs
                    },
                    'plot': [{
                        'type': 'histogram',
                        'variable': 0,
                        'channel': channel,
                        'detectors': [0],
                        'colors': SUBCHANNEL_COLORS,
                        'xlabel': xlabel,
                        'ylabel': 'Area Normalized',
                        'xlim': xlim,
                        'ylim': ylim,
                        'rlim': [0.0, 2.0],
                        'ratio': 'data',
                        'scale_by_width': 'enabled',
                        # Systematics are configured in these XMLs, so the band
                        # is shown; it covers flux, cross section, detector,
                        # POT normalization and MC stat.
                        'disable_systematics': False,
                        'band_label': 'Total Uncertainty',
                        # Percentages, never yields: the samples are blind and
                        # the POT in the configs is bogus.
                        'counter_fmt': '.1%',
                        'watermark': r'$\bf{SBN}$ Analysis-in-Progress',
                        # No POT watermark by design.
                        'show_chi2': True,
                        'chi2_label': r'$\chi^2_{shape}$',
                        'legend_ncol': legend_ncol,
                        'legend_loc': legend_loc,
                    }],
                })
                if SAVE_TO:
                    staged = Path(SAVE_TO) / f'hist_0_0_{CODE_VERSION}.pdf'
                    target = Path(SAVE_TO) / f'dataaccess_{detector}_{variable}_{channel_tokens[channel]}.pdf'
                    shutil.move(staged, target)
                    print(f'{target.name}')
                else:
                    plt.show()
                made += 1
print(f'{made} comparisons rendered')
